# 06 · End-to-end — upload → testgen → optimize → results

In [ ]:

import sys, pathlib
ROOT = pathlib.Path().resolve().parent if pathlib.Path('notebooks').exists() else pathlib.Path().resolve()
sys.path.insert(0, str(ROOT))


In [ ]:
from src.ingest.loader import _Document
from src.ingest.chunker import chunk_documents
from src.testgen.llm import MockLLM
from src.testgen.pipeline import TestGenPipeline
from src.pipeline.runner import run_pipeline
from src.evaluation.ragas_eval import evaluate_ragas
from src.optimizer.bfts_loop import BFTSLoop
from src.optimizer.tree_node import Stage
from config import BFTSConfig

docs = [_Document('Alpha uses Rust. Bob leads Alpha. Beta uses Go. Alice leads Beta.' * 20,
        {'source_file':f'd{i}.txt','page_number':0,'file_type':'txt'}) for i in range(3)]
pipe = TestGenPipeline(llm=MockLLM(), target_size=10, groundedness_threshold=0.5)
df = pipe.generate(docs)
print('testset rows:', len(df))


In [ ]:
llm = MockLLM()
def run_fn(cfg, documents, queries):
    return run_pipeline(cfg, documents, queries, llm=llm)
def eval_fn(results):
    return evaluate_ragas(results)

loop = BFTSLoop(documents=docs, testset=df, run_fn=run_fn, eval_fn=eval_fn,
                bfts_config=BFTSConfig(num_seeds=2, max_steps=6,
                    stage_budgets={Stage.PRELIMINARY:2, Stage.BASELINE:2, Stage.EXPLORATION:2, Stage.ABLATION:2}))
summary = loop.run()
print('best:', summary['best_config'])
print('score:', summary['best_score'])
print('ablations:', len(summary['ablation_report']))
